In [40]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
import os


In [41]:
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("GROQ_MODEL")

In [42]:
model = ChatGroq(model=model_name, temperature=0) 

In [43]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    rating: str

In [44]:
def create_outline(state:BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on this topic {title}'
    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state

In [45]:
def create_blog(state:BlogState) -> BlogState:

    #fetch title
    title = state['title']

    #fetch outline
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [46]:
def evaluate_blog(state:BlogState) -> BlogState:
    outline = state['outline']

    prompt = f'Based on this outline {outline}, rate my blog out of 5 and generate a score'

    rating = model.invoke(prompt).content
    state['rating'] = rating
    return state

In [47]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)


#edges

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()


In [48]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '## Blog Outline  \n**Title:** *Rise of AI in India: From Startup Hubs to Smart Nation*  \n\n---\n\n### 1. Introduction  \n- **Hook:** A striking statistic or anecdote (e.g., “India’s AI market is projected to hit $10\u202fB by 2025”).  \n- **Purpose of the post:** Explain why the AI boom matters for India’s economy, society, and global standing.  \n- **Road‑map:** Briefly outline the sections that will follow.\n\n---\n\n### 2. Historical Context  \n- **Early AI research in India (1970s‑1990s)**  \n  - Pioneering work at IITs, BITS, and the Indian Institute of Science.  \n  - First AI labs and government‑funded projects.  \n- **The “AI Winter” and resurgence**  \n  - Global slowdown in the 2000s.  \n  - Re‑ignition through digital transformation and mobile penetration.  \n\n---\n\n### 3. Current Landscape of AI in India  \n- **Market Size & Growth**  \n  - CAGR, projected valuation, comparison with other emerging markets.  \n- **Key Sectors L

In [49]:
print(final_state['rating'])

**Blog Outline Rating: 4.6 / 5**

**Why this score?**

| Criterion | Strength | Room for Improvement |
|-----------|----------|----------------------|
| **Comprehensiveness** | Covers every angle – history, market, policy, academia, startups, challenges, future, comparison, CTA, resources. | A few sections (e.g., “Future Outlook” and “Comparative Analysis”) could be merged or tightened to avoid redundancy. |
| **Structure & Flow** | Logical progression from intro → context → landscape → drivers → policy → academia → startups → challenges → outlook → comparison → CTA. | The “FAQ” section is optional; if included, it should be placed after the CTA to keep the narrative linear. |
| **Depth & Detail** | Each subsection lists concrete examples, metrics, and actionable insights. | Some bullet points could benefit from a brief data point or statistic to anchor the claim (e.g., exact CAGR figures). |
| **Audience Adaptability** | Explicit note on tailoring depth for tech enthusiasts, policymak